In [1]:
import requests
import polars as pl
import json
from pprint import pprint
from zoneinfo import ZoneInfo
from datetime import datetime

# ── Config ──────────────────────────────────────────────
LAT      = 24.8607
LON      = 67.0011
TIMEZONE = 'Asia/Karachi'
DAYS     = 7

WEATHER_URL = 'https://api.open-meteo.com/v1/forecast'
AQI_URL     = 'https://air-quality-api.open-meteo.com/v1/air-quality'

In [2]:
weather_params = {
    'latitude':      LAT,
    'longitude':     LON,
    'timezone':      TIMEZONE,
    'forecast_days': DAYS,
    'hourly': ','.join([
        'temperature_2m',
        'relativehumidity_2m',
        'dewpoint_2m',
        'apparent_temperature',
        'precipitation',
        'rain',
        'snowfall',
        'weathercode',
        'pressure_msl',
        'surface_pressure',
        'cloudcover',
        'visibility',
        'windspeed_10m',
        'winddirection_10m',
        'windgusts_10m',
        'uv_index',
        'is_day',
    ])
}

aqi_params = {
    'latitude':      LAT,
    'longitude':     LON,
    'timezone':      TIMEZONE,
    'forecast_days': DAYS,
    'hourly': ','.join([
        'pm10',
        'pm2_5',
        'carbon_monoxide',
        'nitrogen_dioxide',
        'sulphur_dioxide',
        'ozone',
        'aerosol_optical_depth',
        'dust',
        'uv_index',
        'european_aqi',
        'us_aqi',
    ])
}

weather_resp = requests.get(WEATHER_URL, params=weather_params, timeout=30)
aqi_resp     = requests.get(AQI_URL,     params=aqi_params,     timeout=30)

weather_resp.raise_for_status()
aqi_resp.raise_for_status()

weather_raw = weather_resp.json()
aqi_raw     = aqi_resp.json()

print('Weather status:', weather_resp.status_code)
print('AQI status:    ', aqi_resp.status_code)

Weather status: 200
AQI status:     200


In [3]:
print('=== WEATHER TOP-LEVEL KEYS ===')
pprint(list(weather_raw.keys()))

print('\n=== WEATHER METADATA ===')
for k in ['latitude', 'longitude', 'elevation', 'timezone', 'timezone_abbreviation', 'utc_offset_seconds']:
    print(f'  {k}: {weather_raw.get(k)}')

print('\n=== WEATHER HOURLY FIELDS ===')
pprint(list(weather_raw['hourly'].keys()))

print('\n=== WEATHER UNITS ===')
pprint(weather_raw['hourly_units'])

print('\n=== AQI HOURLY FIELDS ===')
pprint(list(aqi_raw['hourly'].keys()))

print('\n=== AQI UNITS ===')
pprint(aqi_raw['hourly_units'])

=== WEATHER TOP-LEVEL KEYS ===
['latitude',
 'longitude',
 'generationtime_ms',
 'utc_offset_seconds',
 'timezone',
 'timezone_abbreviation',
 'elevation',
 'hourly_units',
 'hourly']

=== WEATHER METADATA ===
  latitude: 24.850615
  longitude: 66.992485
  elevation: 7.0
  timezone: Asia/Karachi
  timezone_abbreviation: GMT+5
  utc_offset_seconds: 18000

=== WEATHER HOURLY FIELDS ===
['time',
 'temperature_2m',
 'relativehumidity_2m',
 'dewpoint_2m',
 'apparent_temperature',
 'precipitation',
 'rain',
 'snowfall',
 'weathercode',
 'pressure_msl',
 'surface_pressure',
 'cloudcover',
 'visibility',
 'windspeed_10m',
 'winddirection_10m',
 'windgusts_10m',
 'uv_index',
 'is_day']

=== WEATHER UNITS ===
{'apparent_temperature': '°C',
 'cloudcover': '%',
 'dewpoint_2m': '°C',
 'is_day': '',
 'precipitation': 'mm',
 'pressure_msl': 'hPa',
 'rain': 'mm',
 'relativehumidity_2m': '%',
 'snowfall': 'cm',
 'surface_pressure': 'hPa',
 'temperature_2m': '°C',
 'time': 'iso8601',
 'uv_index': '',
 '

In [4]:
print('=== FIRST WEATHER RECORD ===')
pprint({k: v[0] for k, v in weather_raw['hourly'].items()})

print('\n=== FIRST AQI RECORD ===')
pprint({k: v[0] for k, v in aqi_raw['hourly'].items()})

=== FIRST WEATHER RECORD ===
{'apparent_temperature': 36.1,
 'cloudcover': 7,
 'dewpoint_2m': 27.5,
 'is_day': 0,
 'precipitation': 0.0,
 'pressure_msl': 1002.2,
 'rain': 0.0,
 'relativehumidity_2m': 88,
 'snowfall': 0.0,
 'surface_pressure': 1001.4,
 'temperature_2m': 29.7,
 'time': '2026-06-09T00:00',
 'uv_index': 0.0,
 'visibility': 14860.0,
 'weathercode': 95,
 'winddirection_10m': 224,
 'windgusts_10m': 28.8,
 'windspeed_10m': 13.6}

=== FIRST AQI RECORD ===
{'aerosol_optical_depth': 0.4,
 'carbon_monoxide': 190.0,
 'dust': 31.0,
 'european_aqi': 68,
 'nitrogen_dioxide': 13.3,
 'ozone': 48.0,
 'pm10': 44.4,
 'pm2_5': 21.9,
 'sulphur_dioxide': 5.9,
 'time': '2026-06-09T00:00',
 'us_aqi': 83,
 'uv_index': 0.0}


In [5]:
# Polars reads the hourly dict directly — each key becomes a column
df_weather = pl.DataFrame(weather_raw['hourly'])
df_aqi     = pl.DataFrame(aqi_raw['hourly'])

# Parse time strings → Polars Datetime, then localize to Karachi
df_weather = df_weather.with_columns(
    pl.col('time')
      .str.strptime(pl.Datetime, '%Y-%m-%dT%H:%M')
      .dt.replace_time_zone(TIMEZONE)
      .alias('time')
)

df_aqi = df_aqi.with_columns(
    pl.col('time')
      .str.strptime(pl.Datetime, '%Y-%m-%dT%H:%M')
      .dt.replace_time_zone(TIMEZONE)
      .alias('time')
)

print('Weather shape:', df_weather.shape)
print('AQI shape:    ', df_aqi.shape)
print('\nWeather time range:', df_weather['time'].min(), '→', df_weather['time'].max())
print('AQI time range:    ', df_aqi['time'].min(),     '→', df_aqi['time'].max())

Weather shape: (168, 18)
AQI shape:     (168, 12)

Weather time range: 2026-06-09 00:00:00+05:00 → 2026-06-15 23:00:00+05:00
AQI time range:     2026-06-09 00:00:00+05:00 → 2026-06-15 23:00:00+05:00


In [6]:
df_weather.head()

time,temperature_2m,relativehumidity_2m,dewpoint_2m,apparent_temperature,precipitation,rain,snowfall,weathercode,pressure_msl,surface_pressure,cloudcover,visibility,windspeed_10m,winddirection_10m,windgusts_10m,uv_index,is_day
"datetime[μs, Asia/Karachi]",f64,i64,f64,f64,f64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64
2026-06-09 00:00:00 PKT,29.7,88,27.5,36.1,0.0,0.0,0.0,95,1002.2,1001.4,7,14860.0,13.6,224,28.8,0.0,0
2026-06-09 01:00:00 PKT,29.6,88,27.4,35.9,0.0,0.0,0.0,95,1001.6,1000.8,9,14580.0,14.0,225,31.7,0.0,0
2026-06-09 02:00:00 PKT,29.4,90,27.5,35.7,0.0,0.0,0.0,95,1001.1,1000.3,6,14780.0,14.7,229,32.4,0.0,0
2026-06-09 03:00:00 PKT,29.2,91,27.6,35.6,0.0,0.0,0.0,95,1001.0,1000.2,6,12620.0,14.8,231,32.0,0.0,0
2026-06-09 04:00:00 PKT,29.2,91,27.6,35.6,0.0,0.0,0.0,95,1000.8,1000.0,2,12900.0,14.2,231,32.8,0.0,0


In [7]:
df_aqi.head()

time,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,european_aqi,us_aqi
"datetime[μs, Asia/Karachi]",f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
2026-06-09 00:00:00 PKT,44.4,21.9,190.0,13.3,5.9,48.0,0.4,31.0,0.0,68,83
2026-06-09 01:00:00 PKT,41.7,20.5,159.0,10.2,5.1,51.0,0.41,29.0,0.0,68,82
2026-06-09 02:00:00 PKT,38.0,18.8,137.0,7.6,4.5,54.0,0.41,27.0,0.0,68,81
2026-06-09 03:00:00 PKT,33.9,17.2,126.0,6.1,4.1,55.0,0.4,24.0,0.0,67,81
2026-06-09 04:00:00 PKT,31.3,15.9,123.0,5.0,3.9,57.0,0.39,21.0,0.0,67,80


In [8]:
print('=== WEATHER SCHEMA ===')
print(df_weather.schema)

print('\n=== AQI SCHEMA ===')
print(df_aqi.schema)

=== WEATHER SCHEMA ===
Schema({'time': Datetime(time_unit='us', time_zone='Asia/Karachi'), 'temperature_2m': Float64, 'relativehumidity_2m': Int64, 'dewpoint_2m': Float64, 'apparent_temperature': Float64, 'precipitation': Float64, 'rain': Float64, 'snowfall': Float64, 'weathercode': Int64, 'pressure_msl': Float64, 'surface_pressure': Float64, 'cloudcover': Int64, 'visibility': Float64, 'windspeed_10m': Float64, 'winddirection_10m': Int64, 'windgusts_10m': Float64, 'uv_index': Float64, 'is_day': Int64})

=== AQI SCHEMA ===
Schema({'time': Datetime(time_unit='us', time_zone='Asia/Karachi'), 'pm10': Float64, 'pm2_5': Float64, 'carbon_monoxide': Float64, 'nitrogen_dioxide': Float64, 'sulphur_dioxide': Float64, 'ozone': Float64, 'aerosol_optical_depth': Float64, 'dust': Float64, 'uv_index': Float64, 'european_aqi': Int64, 'us_aqi': Int64})


In [9]:
def null_report(df: pl.DataFrame, name: str) -> pl.DataFrame:
    total = len(df)
    report = (
        df.null_count()
          .unpivot(variable_name='column', value_name='nulls')
          .with_columns(
              (pl.col('nulls') / total * 100).round(1).alias('null_%')
          )
          .filter(pl.col('nulls') > 0)
          .sort('nulls', descending=True)
    )
    print(f'=== {name} NULL REPORT (non-zero only) ===')
    if report.is_empty():
        print('No nulls found')
    else:
        print(report)
    return report

null_w = null_report(df_weather, 'WEATHER')
print()
null_a = null_report(df_aqi, 'AQI')

=== WEATHER NULL REPORT (non-zero only) ===
No nulls found

=== AQI NULL REPORT (non-zero only) ===
shape: (11, 3)
┌───────────────────────┬───────┬────────┐
│ column                ┆ nulls ┆ null_% │
│ ---                   ┆ ---   ┆ ---    │
│ str                   ┆ u32   ┆ f64    │
╞═══════════════════════╪═══════╪════════╡
│ pm10                  ┆ 42    ┆ 25.0   │
│ pm2_5                 ┆ 42    ┆ 25.0   │
│ carbon_monoxide       ┆ 42    ┆ 25.0   │
│ nitrogen_dioxide      ┆ 42    ┆ 25.0   │
│ sulphur_dioxide       ┆ 42    ┆ 25.0   │
│ …                     ┆ …     ┆ …      │
│ aerosol_optical_depth ┆ 42    ┆ 25.0   │
│ dust                  ┆ 42    ┆ 25.0   │
│ uv_index              ┆ 42    ┆ 25.0   │
│ european_aqi          ┆ 41    ┆ 24.4   │
│ us_aqi                ┆ 41    ┆ 24.4   │
└───────────────────────┴───────┴────────┘


In [10]:
print('=== WEATHER DESCRIBE ===')
print(df_weather.describe())

=== WEATHER DESCRIBE ===
shape: (9, 19)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬──────────┬──────────┐
│ statistic  ┆ time      ┆ temperatu ┆ relativeh ┆ … ┆ winddirec ┆ windgusts ┆ uv_index ┆ is_day   │
│ ---        ┆ ---       ┆ re_2m     ┆ umidity_2 ┆   ┆ tion_10m  ┆ _10m      ┆ ---      ┆ ---      │
│ str        ┆ str       ┆ ---       ┆ m         ┆   ┆ ---       ┆ ---       ┆ f64      ┆ f64      │
│            ┆           ┆ f64       ┆ ---       ┆   ┆ f64       ┆ f64       ┆          ┆          │
│            ┆           ┆           ┆ f64       ┆   ┆           ┆           ┆          ┆          │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪══════════╪══════════╡
│ count      ┆ 168       ┆ 168.0     ┆ 168.0     ┆ … ┆ 168.0     ┆ 168.0     ┆ 168.0    ┆ 168.0    │
│ null_count ┆ 0         ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0      ┆ 0.0      │
│ mean       ┆ 2026-06-1 ┆ 31.36131  ┆ 74.982143 ┆ 

In [11]:
print('=== AQI DESCRIBE ===')
print(df_aqi.describe())

=== AQI DESCRIBE ===
shape: (9, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬───────────┐
│ statistic ┆ time      ┆ pm10      ┆ pm2_5     ┆ … ┆ dust      ┆ uv_index ┆ european_ ┆ us_aqi    │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---      ┆ aqi       ┆ ---       │
│ str       ┆ str       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64      ┆ ---       ┆ f64       │
│           ┆           ┆           ┆           ┆   ┆           ┆          ┆ f64       ┆           │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪═══════════╡
│ count     ┆ 168       ┆ 126.0     ┆ 126.0     ┆ … ┆ 126.0     ┆ 126.0    ┆ 127.0     ┆ 127.0     │
│ null_coun ┆ 0         ┆ 42.0      ┆ 42.0      ┆ … ┆ 42.0      ┆ 42.0     ┆ 41.0      ┆ 41.0      │
│ t         ┆           ┆           ┆           ┆   ┆           ┆          ┆           ┆           │
│ mean      ┆ 2026-06-1 ┆ 66.622222 ┆ 25.981746 ┆ … ┆ 6

In [12]:
weather_times = set(df_weather['time'].to_list())
aqi_times     = set(df_aqi['time'].to_list())

only_weather = weather_times - aqi_times
only_aqi     = aqi_times - weather_times

print(f'Weather timestamps : {len(weather_times)}')
print(f'AQI timestamps     : {len(aqi_times)}')
print(f'Only in weather    : {len(only_weather)}')
print(f'Only in AQI        : {len(only_aqi)}')

if only_weather:
    print('\nMismatched weather times:', sorted(only_weather)[:5])
if only_aqi:
    print('\nMismatched AQI times:', sorted(only_aqi)[:5])

Weather timestamps : 168
AQI timestamps     : 168
Only in weather    : 0
Only in AQI        : 0


In [13]:
print('Weather time dtype   :', df_weather['time'].dtype)
print('utc_offset_seconds   :', weather_raw.get('utc_offset_seconds'), '(+5h for Karachi)')

# Show local vs UTC for first record
first_local = df_weather['time'][0]
first_utc   = df_weather.with_columns(
    pl.col('time').dt.convert_time_zone('UTC').alias('time_utc')
)['time_utc'][0]

print(f'\nFirst record (local): {first_local}')
print(f'First record (UTC)  : {first_utc}')

Weather time dtype   : Datetime(time_unit='us', time_zone='Asia/Karachi')
utc_offset_seconds   : 18000 (+5h for Karachi)

First record (local): 2026-06-09 00:00:00+05:00
First record (UTC)  : 2026-06-08 19:00:00+00:00


In [14]:
weather_diffs = (
    df_weather
    .with_columns(pl.col('time').diff().alias('delta'))
    .filter(pl.col('delta').is_not_null())
    ['delta']
    .unique()
)

aqi_diffs = (
    df_aqi
    .with_columns(pl.col('time').diff().alias('delta'))
    .filter(pl.col('delta').is_not_null())
    ['delta']
    .unique()
)

print('Weather time deltas (unique):', weather_diffs.to_list())
print('AQI time deltas (unique):    ', aqi_diffs.to_list())
print('\nGrain is uniform hourly:',
    len(weather_diffs) == 1 and len(aqi_diffs) == 1)

Weather time deltas (unique): [datetime.timedelta(seconds=3600)]
AQI time deltas (unique):     [datetime.timedelta(seconds=3600)]

Grain is uniform hourly: True


In [15]:
# Join is_day from weather into AQI for cross-check
df_merged = df_aqi.join(
    df_weather.select(['time', 'is_day']),
    on='time',
    how='inner'
)

aqi_cols = ['pm2_5', 'pm10', 'us_aqi', 'european_aqi', 'uv_index']

for col in aqi_cols:
    if col in df_merged.columns:
        day_nulls   = df_merged.filter(pl.col('is_day') == 1)[col].null_count()
        night_nulls = df_merged.filter(pl.col('is_day') == 0)[col].null_count()
        print(f'{col:25s}  day_nulls={day_nulls}  night_nulls={night_nulls}')

pm2_5                      day_nulls=28  night_nulls=14
pm10                       day_nulls=28  night_nulls=14
us_aqi                     day_nulls=27  night_nulls=14
european_aqi               day_nulls=27  night_nulls=14
uv_index                   day_nulls=28  night_nulls=14


In [16]:
df_weather.glimpse()

Rows: 168
Columns: 18
$ time                 <datetime[μs, Asia/Karachi]> 2026-06-09 00:00:00+05:00, 2026-06-09 01:00:00+05:00, 2026-06-09 02:00:00+05:00, 2026-06-09 03:00:00+05:00, 2026-06-09 04:00:00+05:00, 2026-06-09 05:00:00+05:00, 2026-06-09 06:00:00+05:00, 2026-06-09 07:00:00+05:00, 2026-06-09 08:00:00+05:00, 2026-06-09 09:00:00+05:00
$ temperature_2m                              <f64> 29.7, 29.6, 29.4, 29.2, 29.2, 29.2, 29.3, 30.0, 31.1, 32.0
$ relativehumidity_2m                         <i64> 88, 88, 90, 91, 91, 91, 90, 85, 77, 73
$ dewpoint_2m                                 <f64> 27.5, 27.4, 27.5, 27.6, 27.6, 27.6, 27.4, 27.2, 26.6, 26.5
$ apparent_temperature                        <f64> 36.1, 35.9, 35.7, 35.6, 35.6, 35.4, 35.3, 35.4, 35.9, 36.8
$ precipitation                               <f64> 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
$ rain                                        <f64> 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
$ snowfall                     

In [17]:
df_aqi.glimpse()

Rows: 168
Columns: 12
$ time                  <datetime[μs, Asia/Karachi]> 2026-06-09 00:00:00+05:00, 2026-06-09 01:00:00+05:00, 2026-06-09 02:00:00+05:00, 2026-06-09 03:00:00+05:00, 2026-06-09 04:00:00+05:00, 2026-06-09 05:00:00+05:00, 2026-06-09 06:00:00+05:00, 2026-06-09 07:00:00+05:00, 2026-06-09 08:00:00+05:00, 2026-06-09 09:00:00+05:00
$ pm10                                         <f64> 44.4, 41.7, 38.0, 33.9, 31.3, 29.9, 28.2, 29.0, 32.9, 36.8
$ pm2_5                                        <f64> 21.9, 20.5, 18.8, 17.2, 15.9, 15.2, 14.6, 15.2, 17.7, 19.4
$ carbon_monoxide                              <f64> 190.0, 159.0, 137.0, 126.0, 123.0, 136.0, 180.0, 240.0, 275.0, 261.0
$ nitrogen_dioxide                             <f64> 13.3, 10.2, 7.6, 6.1, 5.0, 4.4, 5.6, 7.3, 8.1, 7.2
$ sulphur_dioxide                              <f64> 5.9, 5.1, 4.5, 4.1, 3.9, 3.7, 4.1, 4.7, 5.1, 5.2
$ ozone                                        <f64> 48.0, 51.0, 54.0, 55.0, 57.0, 59.0, 57.0, 55.0, 56.

# For Pakistan

In [18]:
import geonamescache
import requests
import json
from datetime import date, timedelta
from pathlib import Path

# ── Pakistan cities ──────────────────────────────────────
gc = geonamescache.GeonamesCache()
all_cities = gc.get_cities()

END_DATE   = date.today().strftime('%Y-%m-%d')
START_DATE = (date.today() - timedelta(days=7)).strftime('%Y-%m-%d')

pk_cities = [
    {
        'geonameid': gid,
        'city':      city['name'],
        'province':  city['admin1code'],
        'latitude':  float(city['latitude']),
        'longitude': float(city['longitude']),
        'population': city['population'],
        'timezone':  city['timezone'],
    }
    for gid, city in all_cities.items()
    if city['countrycode'] == 'PK'
]

print(f'Total cities to fetch: {len(pk_cities)}')

# ── API params ───────────────────────────────────────────
WEATHER_URL = 'https://api.open-meteo.com/v1/forecast'
AQI_URL     = 'https://air-quality-api.open-meteo.com/v1/air-quality'
DAYS        = 7

WEATHER_FIELDS = ','.join([
    'temperature_2m', 'relativehumidity_2m', 'dewpoint_2m',
    'apparent_temperature', 'precipitation', 'rain', 'snowfall',
    'weathercode', 'pressure_msl', 'surface_pressure', 'cloudcover',
    'visibility', 'windspeed_10m', 'winddirection_10m', 'windgusts_10m',
    'uv_index', 'is_day',
])

AQI_FIELDS = ','.join([
    'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide',
    'sulphur_dioxide', 'ozone', 'aerosol_optical_depth',
    'dust', 'uv_index', 'european_aqi', 'us_aqi',
])

# ── Fetch ────────────────────────────────────────────────
weather_results = {}
aqi_results     = {}
failed          = []

for i, city in enumerate(pk_cities):
    name = city['city']
    lat  = city['latitude']
    lon  = city['longitude']
    tz   = city['timezone']

    print(f'[{i+1}/{len(pk_cities)}] {name}', end=' ... ')

    try:
        w = requests.get(WEATHER_URL, params={
            'latitude':   lat, 'longitude': lon,
            'timezone':   tz,
            'start_date': START_DATE,
            'end_date':   END_DATE,        # today — no future hours
            'hourly':     WEATHER_FIELDS,
        }, timeout=30)
        w.raise_for_status()

        a = requests.get(AQI_URL, params={
            'latitude':   lat, 'longitude': lon,
            'timezone':   tz,
            'start_date': START_DATE,
            'end_date':   END_DATE,
            'hourly':     AQI_FIELDS,
        }, timeout=30)
        a.raise_for_status()

        weather_results[name] = {'meta': city, 'data': w.json()}
        aqi_results[name]     = {'meta': city, 'data': a.json()}
        print('OK')

    except Exception as e:
        print(f'FAILED — {e}')
        failed.append(name)

# ── Save raw JSON ────────────────────────────────────────
Path('raw').mkdir(exist_ok=True)
with open('raw/weather_pakistan.json', 'w') as f:
    json.dump(weather_results, f, indent=2)
with open('raw/aqi_pakistan.json', 'w') as f:
    json.dump(aqi_results, f, indent=2)

print(f'\nDone. Success: {len(weather_results)}  Failed: {len(failed)}')
if failed:
    print('Failed cities:', failed)

Total cities to fetch: 328
[1/328] Chuhar Jamali ... OK
[2/328] Rawalakot ... OK
[3/328] Pir Jo Goth ... OK
[4/328] Khairpur Mir’s ... OK
[5/328] Zhob ... OK
[6/328] Zaida ... OK
[7/328] Zahir Pir ... OK
[8/328] Zafarwal ... OK
[9/328] Yazman ... OK
[10/328] Wazirabad ... OK
[11/328] Chak Five Hundred Seventy-five ... OK
[12/328] Warah ... OK
[13/328] Vihari ... OK
[14/328] Utmanzai ... OK
[15/328] Uthal ... OK
[16/328] Usta Muhammad ... OK
[17/328] Umarkot ... OK
[18/328] Ubauro ... OK
[19/328] Turbat ... OK
[20/328] Tordher ... OK
[21/328] Topi ... OK
[22/328] Toba Tek Singh ... OK
[23/328] Thul ... OK
[24/328] Thatta ... OK
[25/328] Tharu Shah ... OK
[26/328] Taunsa ... OK
[27/328] Tank ... OK
[28/328] Tangi ... OK
[29/328] Tando Muhammad Khan ... OK
[30/328] Tando Jam ... OK
[31/328] Tando Allahyar ... OK
[32/328] Tando Adam ... OK
[33/328] Tandlianwala ... OK
[34/328] Talhar ... OK
[35/328] Talamba ... OK
[36/328] Talagang ... OK
[37/328] Thal ... OK
[38/328] Swabi ... OK
[39/328]

In [34]:
with open('raw/weather_pakistan.json') as f:
    raw = json.load(f)

dfs = []
for city_name, payload in raw.items():
    meta   = payload['meta']
    hourly = payload['data']['hourly']

    df = (
        pl.DataFrame(hourly)
        .with_columns([
            pl.col('time')
              .str.strptime(pl.Datetime, '%Y-%m-%dT%H:%M')
              .dt.replace_time_zone('Asia/Karachi'),
            pl.lit(meta['geonameid']).alias('geonameid'),
            pl.lit(meta['city']).alias('city'),
            pl.lit(meta['province']).alias('province'),
            pl.lit(meta['latitude']).alias('latitude'),
            pl.lit(meta['longitude']).alias('longitude'),
            pl.lit(meta['timezone']).alias('timezone'),
            pl.lit(meta['population']).alias('population'),
        ])
    )
    dfs.append(df)

weather_df = pl.concat(dfs)

with open(r'raw\aqi_pakistan.json') as f:
    raw = json.load(f)

dfs = []
for city_name, payload in raw.items():
    meta   = payload['meta']
    hourly = payload['data']['hourly']

    df = (
        pl.DataFrame(hourly)
        .with_columns([
            pl.col('time')
              .str.strptime(pl.Datetime, '%Y-%m-%dT%H:%M')
              .dt.replace_time_zone('Asia/Karachi'),
            pl.lit(meta['geonameid']).alias('geonameid'),
            pl.lit(meta['city']).alias('city'),
            pl.lit(meta['province']).alias('province'),
            pl.lit(meta['latitude']).alias('latitude'),
            pl.lit(meta['longitude']).alias('longitude'),
            pl.lit(meta['timezone']).alias('timezone'),
            pl.lit(meta['population']).alias('population'),
        ])
    )
    dfs.append(df)

aqi_df = pl.concat(dfs)

In [35]:
print(weather_df.glimpse())
print("Samples")
print(weather_df.head())

Rows: 62592
Columns: 25
$ time                 <datetime[μs, Asia/Karachi]> 2026-06-02 00:00:00+05:00, 2026-06-02 01:00:00+05:00, 2026-06-02 02:00:00+05:00, 2026-06-02 03:00:00+05:00, 2026-06-02 04:00:00+05:00, 2026-06-02 05:00:00+05:00, 2026-06-02 06:00:00+05:00, 2026-06-02 07:00:00+05:00, 2026-06-02 08:00:00+05:00, 2026-06-02 09:00:00+05:00
$ temperature_2m                              <f64> 29.4, 29.1, 29.0, 28.5, 28.4, 28.2, 28.3, 29.5, 31.1, 33.0
$ relativehumidity_2m                         <i64> 76, 79, 80, 82, 83, 85, 85, 78, 70, 61
$ dewpoint_2m                                 <f64> 24.8, 25.0, 25.1, 25.2, 25.3, 25.4, 25.5, 25.3, 25.1, 24.6
$ apparent_temperature                        <f64> 34.1, 35.0, 34.2, 33.5, 33.8, 33.9, 33.6, 34.4, 35.8, 37.3
$ precipitation                               <f64> 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
$ rain                                        <f64> 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
$ snowfall                   

In [36]:
print(aqi_df.glimpse())
print("Samples")
print(aqi_df.head())

Rows: 62592
Columns: 19
$ time                  <datetime[μs, Asia/Karachi]> 2026-06-02 00:00:00+05:00, 2026-06-02 01:00:00+05:00, 2026-06-02 02:00:00+05:00, 2026-06-02 03:00:00+05:00, 2026-06-02 04:00:00+05:00, 2026-06-02 05:00:00+05:00, 2026-06-02 06:00:00+05:00, 2026-06-02 07:00:00+05:00, 2026-06-02 08:00:00+05:00, 2026-06-02 09:00:00+05:00
$ pm10                                         <f64> 99.0, 78.0, 70.5, 70.1, 74.0, 88.7, 88.0, 80.6, 63.1, 102.2
$ pm2_5                                        <f64> 32.0, 28.0, 26.3, 24.8, 24.8, 26.4, 26.1, 25.5, 25.4, 28.1
$ carbon_monoxide                              <f64> 118.0, 109.0, 102.0, 102.0, 105.0, 107.0, 107.0, 106.0, 106.0, 106.0
$ nitrogen_dioxide                             <f64> 1.3, 1.6, 1.7, 1.7, 1.5, 1.8, 1.4, 0.9, 0.5, 0.3
$ sulphur_dioxide                              <f64> 0.6, 0.5, 0.5, 0.5, 0.5, 0.6, 0.6, 0.6, 0.6, 0.6
$ ozone                                        <f64> 49.0, 47.0, 45.0, 44.0, 45.0, 45.0, 49.0, 54.0, 58

In [ ]:
province_map = {
    '02': 'Balochistan',
    '03': 'Khyber Pakhtunkhwa',
    '04': 'Punjab',
    '05': 'Sindh',
    '06': 'Azad Kashmir',
    '07': 'Gilgit-Baltistan',
    '08': 'Islamabad Capital Territory',
}

season_map = {
    1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring',
    5: 'Summer', 6: 'Summer',
    7: 'Monsoon', 8: 'Monsoon', 9: 'Monsoon',
    10: 'Autumn', 11: 'Autumn',
    12: 'Winter'
}

day_name_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday',
                3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}

month_name_map = {1: 'January', 2: 'February', 3: 'March', 4: 'April',
                  5: 'May', 6: 'June', 7: 'July', 8: 'August',
                  9: 'September', 10: 'October', 11: 'November', 12: 'December'}

dim_location = (
    weather_df.select(['geonameid', 'city', 'province', 'latitude', 'longitude', 'population', 'timezone'])
    .unique(subset=['geonameid'])
    .with_columns([
        pl.col('province').replace(province_map).alias('province_name'),
        pl.col('province').alias('province_code'),
    ])
    .drop('province')
    .rename({'city': 'city_name'})
    .with_row_index('location_id', offset=1)
)

print('dim_location:', dim_location.shape)
print(dim_location.columns)
print(dim_location.head(3))

dim_location: (326, 9)
['location_id', 'geonameid', 'city_name', 'latitude', 'longitude', 'population', 'timezone', 'province_name', 'province_code']
shape: (3, 9)
┌───────────┬───────────┬───────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ location_ ┆ geonameid ┆ city_name ┆ latitude ┆ … ┆ populatio ┆ timezone  ┆ province_ ┆ province_ │
│ id        ┆ ---       ┆ ---       ┆ ---      ┆   ┆ n         ┆ ---       ┆ name      ┆ code      │
│ ---       ┆ str       ┆ str       ┆ f64      ┆   ┆ ---       ┆ str       ┆ ---       ┆ ---       │
│ u32       ┆           ┆           ┆          ┆   ┆ i32       ┆           ┆ str       ┆ str       │
╞═══════════╪═══════════╪═══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1         ┆ 1168718   ┆ Okara     ┆ 30.81029 ┆ … ┆ 533693    ┆ Asia/Kara ┆ Punjab    ┆ 04        │
│           ┆           ┆           ┆          ┆   ┆           ┆ chi       ┆           ┆           │
│ 2         ┆ 1166381   ┆ Sa

In [41]:
dim_time = (
    weather_df.select('time')
    .unique()
    .sort('time')
    .rename({'time': 'ts'})
    .with_columns([
        pl.col('ts').dt.date().alias('date'),
        pl.col('ts').dt.year().cast(pl.Int16).alias('year'),
        pl.col('ts').dt.month().cast(pl.Int16).alias('month'),
        pl.col('ts').dt.day().cast(pl.Int16).alias('day'),
        pl.col('ts').dt.hour().cast(pl.Int16).alias('hour'),
        pl.col('ts').dt.weekday().cast(pl.Int16).alias('day_of_week'),
    ])
    .with_columns([
        pl.col('ts').dt.strftime('%B').alias('month_name'),
        pl.col('ts').dt.strftime('%A').alias('day_name'),
        pl.col('day_of_week').is_in([5, 6]).alias('is_weekend'),
        pl.col('month').map_elements(lambda m: season_map[m], return_dtype=pl.String).alias('season'),
    ])
    .with_row_index('time_id', offset=1)
)

print('\ndim_time:', dim_time.shape)
print(dim_time.columns)
print(dim_time.head(3))


dim_time: (192, 12)
['time_id', 'ts', 'date', 'year', 'month', 'day', 'hour', 'day_of_week', 'month_name', 'day_name', 'is_weekend', 'season']
shape: (3, 12)
┌─────────┬───────────────┬────────────┬──────┬───┬────────────┬──────────┬────────────┬────────┐
│ time_id ┆ ts            ┆ date       ┆ year ┆ … ┆ month_name ┆ day_name ┆ is_weekend ┆ season │
│ ---     ┆ ---           ┆ ---        ┆ ---  ┆   ┆ ---        ┆ ---      ┆ ---        ┆ ---    │
│ u32     ┆ datetime[μs,  ┆ date       ┆ i16  ┆   ┆ str        ┆ str      ┆ bool       ┆ str    │
│         ┆ Asia/Karachi] ┆            ┆      ┆   ┆            ┆          ┆            ┆        │
╞═════════╪═══════════════╪════════════╪══════╪═══╪════════════╪══════════╪════════════╪════════╡
│ 1       ┆ 2026-06-02    ┆ 2026-06-02 ┆ 2026 ┆ … ┆ June       ┆ Tuesday  ┆ false      ┆ Summer │
│         ┆ 00:00:00 PKT  ┆            ┆      ┆   ┆            ┆          ┆            ┆        │
│ 2       ┆ 2026-06-02    ┆ 2026-06-02 ┆ 2026 ┆ … ┆ June 

In [46]:
fact_weather_hourly = (
    weather_df.join(dim_location.select(['geonameid', 'location_id']), on='geonameid', how='left')
    .join(dim_time.select(['ts', 'time_id']), left_on='time', right_on='ts', how='left')
    .drop(['geonameid', 'city', 'province', 'latitude', 'longitude', 'population', 'timezone'])
    .rename({'time': 'ts'})
    .with_columns(pl.col('is_day').cast(pl.Boolean))
    .select([
        'ts', 'location_id', 'time_id',
        'temperature_2m', 'apparent_temperature', 'dewpoint_2m',
        'relativehumidity_2m', 'precipitation', 'rain', 'snowfall',
        'weathercode', 'pressure_msl', 'surface_pressure', 'cloudcover',
        'visibility', 'windspeed_10m', 'winddirection_10m', 'windgusts_10m',
        'uv_index', 'is_day',
    ])
)

print('\nfact_weather_hourly:', fact_weather_hourly.shape)
print(fact_weather_hourly.columns)
print(fact_weather_hourly.head(3))


fact_weather_hourly: (62592, 20)
['ts', 'location_id', 'time_id', 'temperature_2m', 'apparent_temperature', 'dewpoint_2m', 'relativehumidity_2m', 'precipitation', 'rain', 'snowfall', 'weathercode', 'pressure_msl', 'surface_pressure', 'cloudcover', 'visibility', 'windspeed_10m', 'winddirection_10m', 'windgusts_10m', 'uv_index', 'is_day']
shape: (3, 20)
┌────────────┬────────────┬─────────┬────────────┬───┬────────────┬────────────┬──────────┬────────┐
│ ts         ┆ location_i ┆ time_id ┆ temperatur ┆ … ┆ winddirect ┆ windgusts_ ┆ uv_index ┆ is_day │
│ ---        ┆ d          ┆ ---     ┆ e_2m       ┆   ┆ ion_10m    ┆ 10m        ┆ ---      ┆ ---    │
│ datetime[μ ┆ ---        ┆ u32     ┆ ---        ┆   ┆ ---        ┆ ---        ┆ f64      ┆ bool   │
│ s, Asia/Ka ┆ u32        ┆         ┆ f64        ┆   ┆ i64        ┆ f64        ┆          ┆        │
│ rachi]     ┆            ┆         ┆            ┆   ┆            ┆            ┆          ┆        │
╞════════════╪════════════╪═════════╪══

In [49]:
fact_aqi_hourly = (
    aqi_df
    .join(dim_location.select(['geonameid', 'location_id']), on='geonameid', how='left')
    .join(dim_time.select(['ts', 'time_id']), left_on='time', right_on='ts', how='left')
    .drop(['geonameid', 'city', 'province', 'latitude', 'longitude', 'population', 'timezone'])
    .rename({'time': 'ts'})
    .select([
        'ts', 'location_id', 'time_id',
        'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide',
        'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust',
        'uv_index', 'european_aqi', 'us_aqi',
    ])
)

print('\nfact_aqi_hourly:', fact_aqi_hourly.shape)
print(fact_aqi_hourly.columns)
print(fact_aqi_hourly.head(3))


fact_aqi_hourly: (62592, 14)
['ts', 'location_id', 'time_id', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index', 'european_aqi', 'us_aqi']
shape: (3, 14)
┌───────────────┬─────────────┬─────────┬──────┬───┬───────┬──────────┬──────────────┬────────┐
│ ts            ┆ location_id ┆ time_id ┆ pm10 ┆ … ┆ dust  ┆ uv_index ┆ european_aqi ┆ us_aqi │
│ ---           ┆ ---         ┆ ---     ┆ ---  ┆   ┆ ---   ┆ ---      ┆ ---          ┆ ---    │
│ datetime[μs,  ┆ u32         ┆ u32     ┆ f64  ┆   ┆ f64   ┆ f64      ┆ i64          ┆ i64    │
│ Asia/Karachi] ┆             ┆         ┆      ┆   ┆       ┆          ┆              ┆        │
╞═══════════════╪═════════════╪═════════╪══════╪═══╪═══════╪══════════╪══════════════╪════════╡
│ 2026-06-02    ┆ 166         ┆ 1       ┆ 99.0 ┆ … ┆ 112.0 ┆ 0.0      ┆ 79           ┆ 82     │
│ 00:00:00 PKT  ┆             ┆         ┆      ┆   ┆       ┆          ┆              ┆        │

In [50]:
fact_weather_hourly.select('winddirection_10m')

winddirection_10m
i64
223
176
195
205
219
…
146
81
87


In [1]:
pk_cities_sample = [
    {'geonameid': gid, 'province': city['admin1code'], 'city': city['name']}
    for gid, city in all_cities.items()
    if city['countrycode'] == 'PK'
]
# print unique province codes
unique_codes = set(c['province'] for c in pk_cities_sample)
print(sorted(unique_codes))

NameError: name 'all_cities' is not defined